In [36]:
import torch.nn as nn
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

In [37]:
from neuralnet_arch import Net
# -----------------------------
# Global Parameters
# -----------------------------
NUM_INPUTS = 4
NUM_OUTPUTS = 4
BATCH_SIZE = 64
NUM_EPOCHS = 1000
LEARNING_RATE = 1e-3
VAL_SPLIT = 0.2
RANDOM_SEED = 42
DATA_PATH = "gen_data.txt"  # <-- change this to your data file path

# -----------------------------
# Load and Split Data
# -----------------------------
def load_and_split_data(file_path, val_split=0.2, random_seed=42):
    data = np.loadtxt(file_path, delimiter=",")
    np.random.seed(random_seed)
    indices = np.random.permutation(len(data))
    data = data[indices]

    X = data[:, :NUM_INPUTS]
    y = data[:, NUM_INPUTS:NUM_INPUTS + NUM_OUTPUTS]

    #Log transform x data
    X = np.log10(X)

    n_total = len(X)
    n_val = int(n_total * val_split)
    n_train = n_total - n_val

    X_train, y_train = X[:n_train], y[:n_train]
    X_val, y_val = X[n_train:], y[n_train:]

    return X_train, X_val, y_train, y_val

# -----------------------------
# Training Function
# -----------------------------
def train_model(model, train_loader, val_loader, num_epochs, lr, device,checkpoint_path,patience=20):
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_loss = float('inf')  # Initialize with a large value
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
        
        # Save the model checkpoint if validation loss improved
        if val_loss < best_val_loss:
            print(f"Validation loss decreased ({best_val_loss:.4f} -> {val_loss:.4f}). Saving model...")
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)
            epochs_no_improve = 0  # Reset counter
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s).")

        # Early stopping
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs. Best val loss: {best_val_loss:.4f}")
            break


# -----------------------------
# Main Execution
# -----------------------------
def main():
    # Load and prepare data
    X_train, X_val, y_train, y_val = load_and_split_data(DATA_PATH, val_split=VAL_SPLIT, random_seed=RANDOM_SEED)

    # Convert to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    X_val_tensor   = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor   = torch.tensor(y_val, dtype=torch.float32)

    # Create datasets and loaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset   = TensorDataset(X_val_tensor, y_val_tensor)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # Model
    model = Net(input_dim=NUM_INPUTS, output_dim=NUM_OUTPUTS)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Train
    train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS, lr=LEARNING_RATE, device=device,checkpoint_path="test.pth")
main()

Epoch [1/1000] - Train Loss: 28.7768 - Val Loss: 1.6491
Validation loss decreased (inf -> 1.6491). Saving model...
Epoch [2/1000] - Train Loss: 1.3565 - Val Loss: 1.2645
Validation loss decreased (1.6491 -> 1.2645). Saving model...
Epoch [3/1000] - Train Loss: 1.0547 - Val Loss: 0.9881
Validation loss decreased (1.2645 -> 0.9881). Saving model...
Epoch [4/1000] - Train Loss: 0.8340 - Val Loss: 0.8493
Validation loss decreased (0.9881 -> 0.8493). Saving model...
Epoch [5/1000] - Train Loss: 0.6950 - Val Loss: 0.6907
Validation loss decreased (0.8493 -> 0.6907). Saving model...
Epoch [6/1000] - Train Loss: 0.6261 - Val Loss: 0.6272
Validation loss decreased (0.6907 -> 0.6272). Saving model...
Epoch [7/1000] - Train Loss: 0.5856 - Val Loss: 0.5797
Validation loss decreased (0.6272 -> 0.5797). Saving model...
Epoch [8/1000] - Train Loss: 0.5649 - Val Loss: 0.5404
Validation loss decreased (0.5797 -> 0.5404). Saving model...
Epoch [9/1000] - Train Loss: 0.5156 - Val Loss: 0.4955
Validation 

In [38]:
# -----------------------------
# Load the Best Model
# -----------------------------
def load_model(checkpoint_path, device='cpu'):
    # Initialize the model
    model = Net(input_dim=NUM_INPUTS, output_dim=NUM_OUTPUTS)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)  # Ensure model is on the right device (CPU or GPU)
    model.eval()  # Set model to evaluation mode
    return model

# -----------------------------
# Make Predictions
# -----------------------------
def make_predictions(model, X_input, device='cpu'):
    # Convert the input to a tensor and move to the correct device
    X_input_tensor = torch.tensor(X_input, dtype=torch.float32).to(device)
    
    # Make the prediction (no gradients needed for inference)
    with torch.no_grad():
        predictions = model(X_input_tensor)

    return predictions.cpu().numpy()  # Convert back to NumPy array

# -----------------------------
# Usage
# -----------------------------
# Example path to checkpoint
checkpoint_path = "test.pth"

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(checkpoint_path, device=device)

# Example input (new data to predict on)
# Make sure this input has the same shape as the training data
X_input = np.array([25.0, 1e-14, 1e6, 129])  # Replace with actual data

# Log-transform the input if necessary (since you applied log10 during training)
X_input = np.log10(X_input)

# Make prediction
predictions = make_predictions(model, X_input, device=device)
print("Predictions: ", predictions)

Predictions:  [-2.0972050e+01  9.4067365e-01  3.5403688e-03 -8.1906468e-03]
